In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("01 covid_toy.csv" )
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [3]:
print(df.cough.value_counts())

print()
print()

print(df.city.value_counts())

cough
Mild      62
Strong    38
Name: count, dtype: int64


city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64


In [4]:
'''
age - numeric column
gender - Nominal categorical column(One-Hot encoding)
fever - Numeric column. Has missing values. Will apply simple imputer to fill missing values
cough - Ordinal categorical column(Ordinal encoding)
city - Nominal categorical column(One-Hot encoding)
has_covid - Target column categorical.(Label encoding)

Here, we will work with columns gender, fever, cough and city as discussed above .
'''

'\nage - numeric column\ngender - Nominal categorical column(One-Hot encoding)\nfever - Numeric column. Has missing values. Will apply simple imputer to fill missing values\ncough - Ordinal categorical column(Ordinal encoding)\ncity - Nominal categorical column(One-Hot encoding)\nhas_covid - Target column categorical.(Label encoding)\n\nHere, we will work with columns gender, fever, cough and city as discussed above .\n'

In [5]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [6]:
x = df.iloc[:,0:5]
y = df.drop(x, axis = 'columns')

print(x.head())

print()
print()
print()

print(y.head())

   age  gender  fever cough     city
0   60    Male  103.0  Mild  Kolkata
1   27    Male  100.0  Mild    Delhi
2   42    Male  101.0  Mild    Delhi
3   31  Female   98.0  Mild  Kolkata
4   65  Female  101.0  Mild   Mumbai



  has_covid
0        No
1       Yes
2        No
3        No
4        No


In [7]:
# Train-Test split

from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2)
print(x_train.head())
print()
print()
print(y_train.head())

    age  gender  fever   cough       city
97   20  Female  101.0    Mild  Bangalore
75    5    Male  102.0    Mild    Kolkata
61   81  Female   98.0  Strong     Mumbai
19   42  Female    NaN  Strong  Bangalore
58   23    Male   98.0  Strong     Mumbai


   has_covid
97        No
75       Yes
61        No
19       Yes
58       Yes


# 1) Without column transformer.

In [8]:
# Perfornming SimpleImputer on fever column for filling missing values.

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

In [9]:
si = SimpleImputer()
le = LabelEncoder()

In [10]:
# Applying label encodinng on clas column.

y_train.has_covid = le.fit_transform(y_train.has_covid)
y_test.has_covid = le.transform(y_test['has_covid'])

In [11]:
x_train['fever'] = si.fit_transform(x_train[['fever']])
x_test['fever'] = si.transform(x_test[['fever']])


# Here, we'll get error message bcz 'SimpleImputer' expects 2D array and we are passing 1D  array.

2
2


(80, 1)
(20, 1)


# Ordinal encoding cough

In [13]:
from sklearn.preprocessing import OrdinalEncoder

In [14]:
oe = OrdinalEncoder()

x_train['cough'] = oe.fit_transform(x_train[['cough']])
x_test['cough'] = oe.transform(x_test[['cough']])

# OHE on City and gender

In [15]:
from sklearn.preprocessing import OneHotEncoder

In [16]:
ohe = OneHotEncoder(drop = 'first', sparse_output = False)

In [17]:
x_train_gender_city = ohe.fit_transform(x_train[['gender', 'city']])
x_test_gender_city = ohe.transform(x_test[['gender', 'city']])

print(x_train_gender_city.shape)
print(x_test_gender_city.shape)

(80, 4)
(20, 4)


In [35]:
x_train_gender_city[1]

array([1., 0., 1., 0.])

In [36]:
# Extracting values of age

x_train_age = x_train['age'].values
x_test_age = x_test.age.values

In [37]:
# Concetinating all the arrays

x_train_transformed = np.concatenate((x_train_fever_filled, x_train_cough, x_train_age, x_train_gender_city), axis = 1)
x_test_transformed = np.concatenate((x_test_fever_filled, x_test_cough, x_test_age, x_test_gender_city), axis = 1)

ValueError: all the input arrays must have same number of dimensions, but the array at index 0 has 2 dimension(s) and the array at index 2 has 1 dimension(s)

In [38]:
x_train_fever_filled.ndim

2

In [39]:
x_train_age.ndim

1

In [40]:
x_train_gender_city.ndim

2

In [41]:
x_train_cough.ndim

2

In [42]:
# Reshaping x_train_age to 2D

x_train_age = x_train_age.reshape(-1, 1)

In [43]:
x_train_age.ndim

2

In [44]:
# Reshaping x_test_age to 2D

x_test_age = x_test_age.reshape(-1, 1)

In [45]:
x_test_age.ndim

2

In [46]:
# Concetinating all the arrays

x_train_transformed = np.concatenate((x_train_fever_filled, x_train_cough, x_train_age, x_train_gender_city), axis = 1)
x_test_transformed = np.concatenate((x_test_fever_filled, x_test_cough, x_test_age, x_test_gender_city), axis = 1)

In [47]:
print(x_train_transformed[1])

print()

print(x_test_transformed[1])

[102.   0.   5.   1.   0.   1.   0.]

[100.77631579   0.          84.           0.           0.
   0.           0.        ]


In [48]:
print(x_train_transformed.shape)
print(x_test_transformed.shape)

(80, 7)
(20, 7)


# Column Transformer

In [49]:
from sklearn.compose import ColumnTransformer

# ColumnTransformers() takes two arguments, 
1) `transformers` and
2) `remainder`.

* **remainder**: It's related to those columns we're not performing anything. Means what we wanna do with the left out columns? We wanna `drop` or `passthrough`. `drop` means it will drop such columns on which we have not applied any transformer and `passthrough` means it will keep it as it is.

* **transformers**: This takes, list of tuples. No.of transformer = no.of tuples.
Each tuple takes three values, 1) Name of the transformer, 2) Object of the transformer's class and the columns' names (into a list) we want to apply that particular transformer.

In [51]:
transformer = [('trans1', SimpleImputer(), ['fever']),
               ('trans2', OrdinalEncoder(categories = [['Mild', 'Strong']]), ['cough']),
               ('trans3', OneHotEncoder(sparse_output = False, drop = 'first'), ['gender', 'city'])
               ]

In [ ]:
ct = ColumnTransformer(
    transformers = transformer,
    remainder = 'passthrough').set_output(transform = 'pandas')

In [53]:
x_train_trans = ct.fit_transform(x_train)

In [56]:
x_train_trans.sample(5)

,trans1__fever,trans2__cough,trans3__gender_Male,trans3__city_Delhi,trans3__city_Kolkata,trans3__city_Mumbai,remainder__age
54,99.0,0.0,0.0,0.0,0.0,1.0,60
44,102.0,1.0,1.0,1.0,0.0,0.0,20
48,99.0,1.0,1.0,0.0,0.0,0.0,66
95,104.0,0.0,0.0,0.0,0.0,0.0,12
49,104.0,0.0,1.0,0.0,0.0,1.0,44


In [ ]:
x_train_trans.shape

# We can see that we're getting the same result as we got in above tedious method. 

(80, 7)

In [58]:
x_test_trans = ct.transform(x_test)

In [60]:
print(x_test_trans.sample(5))
print()
print()

print(x_test_trans.shape)

    trans1__fever  trans2__cough  trans3__gender_Male  trans3__city_Delhi  \
29     100.776316            1.0                  0.0                 0.0   
5      100.776316            0.0                  0.0                 0.0   
26     100.000000            0.0                  0.0                 0.0   
19     100.776316            1.0                  0.0                 0.0   
51     100.000000            1.0                  0.0                 0.0   

    trans3__city_Kolkata  trans3__city_Mumbai  remainder__age  
29                   0.0                  1.0              34  
5                    0.0                  0.0              84  
26                   1.0                  0.0              19  
19                   0.0                  0.0              42  
51                   1.0                  0.0              11  


(20, 7)


# `Next Session`:  Day 29 - Machine Learning Pipelines